# Shark Attack EDA (Refactored)

This notebook keeps the analysis narrative in the notebook, while moving reusable loading,
cleaning, and feature engineering logic into `src/`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import prepare_attacks_dataframe

sns.set_theme(style="whitegrid")
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "attacks_cleaned.csv"

## Load and prepare data

In [ ]:
df = prepare_attacks_dataframe(DATA_PATH)
df.head()

In [ ]:
df[[
    "Type",
    "Country",
    "Hemisphere",
    "ocean",
    "Activity",
    "Sex ",
    "Fatal",
    "Classified Species",
    "Date",
    "is_weekend",
    "time_category",
    "Season",
]].sample(10, random_state=42)

## Basic distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

sns.countplot(data=df, x="Fatal", ax=axes[0, 0])
axes[0, 0].set_title("Fatality distribution")

sns.countplot(data=df, x="Activity", order=df["Activity"].value_counts().index, ax=axes[0, 1])
axes[0, 1].tick_params(axis="x", rotation=30)
axes[0, 1].set_title("Activity categories")

sns.countplot(data=df, x="Classified Species", order=df["Classified Species"].value_counts().index, ax=axes[0, 2])
axes[0, 2].tick_params(axis="x", rotation=45)
axes[0, 2].set_title("Shark species buckets")

sns.countplot(data=df, x="time_category", order=df["time_category"].value_counts().index, ax=axes[1, 0])
axes[1, 0].set_title("Time of day")

sns.countplot(data=df, x="ocean", order=df["ocean"].fillna("Missing").value_counts().index, ax=axes[1, 1])
axes[1, 1].tick_params(axis="x", rotation=45)
axes[1, 1].set_title("Ocean")

sns.countplot(data=df, x="Season", order=["Winter", "Spring", "Summer", "Fall"], ax=axes[1, 2])
axes[1, 2].set_title("Season")

plt.tight_layout()

## Fatal vs non-fatal breakdowns

In [ ]:
analysis_df = df[df["Fatal"].isin(["Y", "N"])].copy()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.countplot(data=analysis_df, x="Activity", hue="Fatal", order=analysis_df["Activity"].value_counts().index, ax=axes[0, 0])
axes[0, 0].tick_params(axis="x", rotation=30)
axes[0, 0].set_title("Fatality by activity")

sns.countplot(data=analysis_df, x="Sex ", hue="Fatal", ax=axes[0, 1])
axes[0, 1].set_title("Fatality by sex")

sns.countplot(data=analysis_df, x="Hemisphere", hue="Fatal", ax=axes[1, 0])
axes[1, 0].set_title("Fatality by hemisphere")

sns.countplot(data=analysis_df, x="is_weekend", hue="Fatal", ax=axes[1, 1])
axes[1, 1].set_title("Fatality by weekend")

plt.tight_layout()

## Age distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=analysis_df, x="Age", hue="Fatal", multiple="stack", bins=30, edgecolor="white")
plt.title("Age distribution of fatal vs non-fatal attacks")
plt.show()

## Attack trend over time

In [ ]:
yearly = (
    analysis_df.groupby(["Year", "Fatal"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["N", "Y"], fill_value=0)
)

yearly.plot(kind="bar", stacked=True, figsize=(16, 6))
plt.title("Shark attacks by year and fatality")
plt.ylabel("Count")
plt.show()

## Country-level overview

In [ ]:
by_country = (
    analysis_df.groupby("Country", as_index=False)
    .size()
    .sort_values("size", ascending=False)
)
by_country.head(15)

In [ ]:
fig = px.choropleth(
    by_country,
    locations="Country",
    locationmode="country names",
    color="size",
    color_continuous_scale="Turbo",
    title="Shark attack counts by country",
)
fig.show()